# Crypto.Net · 07 — Mengindeks event ke database

*Dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*

> **Aman dijalankan.** Notebook ini hanya membaca jaringan publik dan menandatangani secara offline; tidak pernah mem-broadcast transaksi. Kunci yang dibuat di sini adalah kunci demo — jangan pernah mengirim dana sungguhan ke sana.

Buka di VS Code dengan ekstensi **Polyglot Notebooks** (atau Jupyter dengan .NET Interactive) lalu jalankan sel secara berurutan. [English version](./07-indexer.en.ipynb)

Yang akan Anda lakukan: men-decode log `Transfer` ERC-20 dari Sepolia, menjalankan `EvmLogIndexer` dari checkpoint ke SQLite dengan Entity Framework Core, meng-query hasilnya dengan LINQ, dan menelusuri blok Polkadot Asset Hub dengan `BlockCrawler`. Semuanya hanya membaca.

In [ ]:
#r "nuget: Crypto.Net.Extensions, 1.2.0"
#r "nuget: Crypto.Net.Indexer.EntityFrameworkCore, 1.2.0"
#r "nuget: Microsoft.EntityFrameworkCore.Sqlite, 10.0.12"

In [ ]:
using System.Numerics;
using Crypto.Net.Core;
using Crypto.Net.Evm;
using Crypto.Net.Indexer;
using Crypto.Net.Indexer.EntityFrameworkCore;
using Crypto.Net.Polkadot;
using Microsoft.Data.Sqlite;
using Microsoft.EntityFrameworkCore;
using Microsoft.EntityFrameworkCore.Infrastructure;

const string Usdc = "0x1c7D4B196Cb0C7B01d743Fbc6116a902379C7238";   // Circle USDC on Sepolia
var sepolia = new EvmRpcClient(EvmChain.Sepolia);
ulong head = await sepolia.GetBlockNumberAsync();
$"Sepolia head {head:N0}"

## Men-decode event

`EvmEvent.Parse` membaca event Solidity yang ditulis manusia. `Decode` mengubah log menjadi nilai bernama: parameter indexed diambil dari topic, sisanya dari data.

In [ ]:
var transfer = EvmEvent.Parse("event Transfer(address indexed from, address indexed to, uint256 value)");
Console.WriteLine($"{transfer.Signature} → topic0 {transfer.Topic0[..18]}…");

var logs = await sepolia.GetLogsAsync(new EvmLogFilter(head - 100, head) { Addresses = [Usdc], Topics = [[transfer.Topic0]] });
foreach (var log in logs.Take(5))
{
    var e = transfer.Decode(log);
    Console.WriteLine($"block {log.BlockNumber}: {e["from"]} → {e["to"]}  {(decimal)(BigInteger)e["value"]! / 1_000_000m} USDC");
}
$"{logs.Count} transfers in the last 100 blocks"

## Mengindeks ke SQLite

Indexer membaca `eth_getLogs` per halaman dari checkpoint sampai head dikurangi `Confirmations`. Jika provider menolak sebuah rentang, indexer membagi dua rentangnya lalu mencoba lagi. Setelah setiap batch ia menulis checkpoint, sehingga restart melanjutkan dari titik terakhir. `EfEvmLogStore` menyimpan log mentah dan transfer ERC-20. Insert-nya idempoten, sehingga batch yang dikirim ulang tidak menimbulkan masalah.

In [ ]:
var connection = new SqliteConnection("Data Source=:memory:");
await connection.OpenAsync();
var factory = new PooledDbContextFactory<CryptoNetIndexerDbContext>(
    new DbContextOptionsBuilder<CryptoNetIndexerDbContext>().UseSqlite(connection).Options);
await using (var db = await factory.CreateDbContextAsync()) await db.Database.EnsureCreatedAsync();

var checkpoints = new EfCheckpointStore<CryptoNetIndexerDbContext>(factory);
var store = new EfEvmLogStore<CryptoNetIndexerDbContext>(factory, new EvmLogStoreOptions { ChainId = "sepolia" });
var indexer = new EvmLogIndexer(sepolia, checkpoints, store, new EvmLogIndexerOptions
{
    Name = "usdc",
    Addresses = [Usdc],
    Events = [EvmEvent.Erc20Transfer],
    StartBlock = head - 500,
    Confirmations = 5,
    MaxBlockRange = 100,
});
ulong? last = await indexer.RunOnceAsync();
$"indexed up to block {last:N0}; checkpoint = {await checkpoints.GetAsync("usdc"):N0}"

In [ ]:
var query = await factory.CreateDbContextAsync();
var top = (await query.TokenTransfers.Where(t => t.ChainId == "sepolia").ToListAsync())
    .GroupBy(t => t.To)
    .Select(g => new { Recipient = g.Key, Transfers = g.Count(), Usdc = g.Sum(t => (decimal)t.RawValue) / 1_000_000m })
    .OrderByDescending(x => x.Usdc)
    .Take(5)
    .ToList();
await query.DisposeAsync();
top

Menjalankan `RunOnceAsync` lagi hanya memproses blok baru, karena checkpoint mengingat titik terakhirnya. Di aplikasi, daftarkan pipeline yang sama dengan `AddEvmLogIndexerToDatabase` dan ia berjalan sebagai hosted service. Template `cnet-worker` melakukan persis hal itu.

## Chain apa pun: `BlockCrawler`

Crawler memanggil handler Anda sekali untuk setiap tinggi blok, berurutan, dengan checkpoint. Di sini ia menghitung event di blok Polkadot Asset Hub.

In [ ]:
var dot = new PolkadotRpcClient(PolkadotChain.PolkadotAssetHub);
ulong dotHead = await dot.GetBlockNumberAsync();
var counts = new List<string>();
var crawler = new BlockCrawler(dot, new InMemoryCheckpointStore(), async (height, ct) =>
{
    var events = await dot.GetEventsAsync(await dot.GetBlockHashAsync(height, ct), ct);
    counts.Add($"#{height}: {events.Count} events, {events.Count(e => e.Is("Balances", "Transfer"))} transfers");
}, new BlockCrawlerOptions { StartBlock = dotHead - 4, Confirmations = 0, MaxBlocksPerPoll = 5 });
await crawler.RunOnceAsync();
string.Join("\n", counts)

In [ ]:
sepolia.Dispose(); dot.Dispose(); connection.Dispose();
"disposed"